In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ============================================================
#  CodeT5-Small LoRA Inference trên tập Spider-Realistic
#  Đảm bảo đồng bộ 100% định dạng Tiền xử lý dữ liệu với File Train
# ============================================================

import os
import shutil
import json
import torch
import re
import time
import nltk
import urllib.request
import zipfile
import numpy as np
from google.colab import drive

# 1. CÀI ĐẶT THƯ VIỆN & CẬP NHẬT TORCHAO TRÁNH XUNG ĐỘT PEFT
!pip install -q "transformers>=4.41.0,<5.0.0" datasets sentencepiece accelerate tqdm peft "torchao>=0.16.0"

from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset

try:
    reconstruct_func = np._core.multiarray._reconstruct if hasattr(np, '_core') else np.core.multiarray._reconstruct
    torch.serialization.add_safe_globals([
        reconstruct_func,
        np.ndarray,
        np.dtype,
    ])
    print("✅ Đã whitelist Numpy cho PyTorch.")
except AttributeError:
    pass

if not hasattr(torch, 'original_load_func'):
    torch.original_load_func = torch.load

def safe_load_override(*args, **kwargs):
    if 'weights_only' in kwargs:
        del kwargs['weights_only']
    return torch.original_load_func(*args, weights_only=False, **kwargs)

torch.load = safe_load_override
print(f"✅ Đã ép buộc torch.load(weights_only=False) thành công.")

# 2. KẾT NỐI DRIVE & ĐƯỜNG DẪN CHECKPOINT LORA CỦA BẠN
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY']      = "0ba946628cb1f5acb76ecd357f590e95"

drive.mount('/content/drive')
FINAL_SAVE_PATH = "/content/drive/MyDrive/CodeT5-small_LoRA"

print(">>> [1/5] Kiểm tra và tải dữ liệu Spider...")
if not os.path.exists('spider_data'):
    os.system("pip install -q kaggle")
    os.system("kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset")
    zip_path = "yale-universitys-spider-10-nlp-dataset.zip"

    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("temp_spider")
        if os.path.exists("temp_spider/spider"):
            shutil.move("temp_spider/spider", "spider_data")
        else:
            shutil.rename("temp_spider", "spider_data")
        shutil.rmtree('temp_spider', ignore_errors=True)
        os.remove(zip_path)

# Tải công cụ chấm điểm chuẩn
if not os.path.exists('evaluation.py'):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py", "evaluation.py")
    urllib.request.urlretrieve("https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py", "process_sql.py")
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Tải tập dữ liệu thực tế Spider-Realistic
if not os.path.exists('spider_data/spider-realistic.json'):
    print("Đang tải tập Spider-Realistic...")
    urllib.request.urlretrieve("https://zenodo.org/records/5205322/files/spider-realistic.json?download=1", "spider_data/spider-realistic.json")

# 3. TIỀN XỬ LÝ ĐỒNG BỘ THEO CONFIG FILE TRAIN (MAPPING TYPE_SHORT)
print(">>> [2/5] Đang xử lý cấu trúc Schema đồng bộ...")

TYPE_SHORT = {"text": "T", "number": "N", "time": "D", "boolean": "B", "others": "O"}

def build_schema_map(tables_path):
    with open(tables_path, encoding='utf-8') as f:
        tables = json.load(f)
    schema_map = {}
    for db in tables:
        db_id     = db['db_id']
        col_types = db.get('column_types', [])
        parts     = []
        for t_idx, t_name in enumerate(db['table_names_original']):
            cols = []
            for c_idx, (t_i, c_name) in enumerate(db['column_names_original']):
                if t_i == t_idx:
                    ct = TYPE_SHORT.get(col_types[c_idx] if c_idx < len(col_types) else "others", "O")
                    cols.append(f"{c_name}:{ct}")
            parts.append(f"{t_name}({','.join(cols)})")
        schema_map[db_id] = " | ".join(parts)
    return schema_map

PREFIX     = "Translate English to SQL: "
SEP        = " | schema: "
PREFIX_TOK = 8
QUESTION_BUDGET = 96
MAX_INPUT_LEN   = 512
MAX_TARGET_LEN  = 256
BEAM_SIZE       = 6

def build_input_smart(question: str, schema: str, tokenizer, max_len=512):
    s_budget = max_len - PREFIX_TOK - QUESTION_BUDGET
    q_ids = tokenizer.encode(question.strip(), add_special_tokens=False, max_length=QUESTION_BUDGET, truncation=True)
    s_ids = tokenizer.encode(schema, add_special_tokens=False, max_length=s_budget, truncation=True)
    q_text = tokenizer.decode(q_ids, skip_special_tokens=True)
    s_text = tokenizer.decode(s_ids, skip_special_tokens=True)
    return f"{PREFIX}{q_text}{SEP}{s_text}"

schema_map = build_schema_map("spider_data/tables.json")

# 4. LOAD MODEL CODET5 + LORA ADAPTERS
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n>>> [3/5] Đang tải mô hình CodeT5 + LoRA lên {device}...")

MODEL_NAME = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Nạp mô hình nền CodeT5 gốc
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
# Ghép các tham số LoRA Adapter đã được huấn luyện tốt nhất từ Drive
model = PeftModel.from_pretrained(base_model, FINAL_SAVE_PATH)
model = model.to(device)
model.eval()

# Nạp và xử lý file Spider-Realistic
with open("spider_data/spider-realistic.json", 'r', encoding='utf-8') as f:
    realistic_data = json.load(f)

print(f"Tìm thấy: {len(realistic_data)} mẫu thử nghiệm trong tập Spider-Realistic.")

# 5. CHẠY INFERENCE (BATCH) THEO THAM SỐ CHUẨN CỦA FILE TRAIN
print("\n>>> [4/5] Bắt đầu sinh câu lệnh SQL (Batch Processing)...")
predictions, gold_lines = [], []
input_texts = []

for item in realistic_data:
    db_id = item['db_id']
    schema = schema_map.get(db_id, "")
    # Sinh text prompt theo chuẩn thông minh đã train
    inp_text = build_input_smart(item['question'], schema, tokenizer, MAX_INPUT_LEN)
    input_texts.append(inp_text)
    gold_lines.append(f"{item['query']}\t{db_id}\n")

batch_size = 16
total_batches = len(input_texts)
start_time = time.time()

for i in range(0, total_batches, batch_size):
    batch_texts = input_texts[i : i + batch_size]

    # max_length đồng bộ theo max_input_len=512 của file train
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_INPUT_LEN)
    input_ids = inputs.input_ids.to(model.device)
    attention_mask = inputs.attention_mask.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_TARGET_LEN,
            num_beams=BEAM_SIZE,           # beam_size = 6 theo cấu hình tối ưu của bạn
            length_penalty=0.8,            # penalty đồng bộ
            early_stopping=True
        )

    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend([pred + "\n" for pred in batch_preds])
    print(f"\rTiến độ: {min(i + batch_size, total_batches)}/{total_batches}", end="")

print(f"\n✅ Quá trình dịch thuật hoàn tất trong {time.time() - start_time:.2f} giây!")

# Ghi kết quả ra file phục vụ chấm điểm
with open('pred.txt', 'w', encoding='utf-8') as f: f.writelines(predictions)
with open('gold.txt', 'w', encoding='utf-8') as f: f.writelines(gold_lines)

# 6. CHẠY ĐÁNH GIÁ CHÍNH THỨC QUA SCRIPT SPIDER
print("\n>>> [5/5] KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP SPIDER-REALISTIC:")
with open("evaluation.py", "r", encoding="utf-8") as f:
    eval_content = f.read()
eval_content = eval_content.replace(
    'conn = sqlite3.connect(db)',
    'conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")'
)
with open("evaluation.py", "w", encoding="utf-8") as f:
    f.write(eval_content)

os.system(
    "python evaluation.py "
    "--gold gold.txt --pred pred.txt "
    "--db spider_data/database "
    "--table spider_data/tables.json "
    "--etype all"
)

# Backup kết quả về Drive để đối chiếu hiệu năng
shutil.copy('pred.txt', os.path.join(FINAL_SAVE_PATH, 'pred_spider_realistic.txt'))
shutil.copy('gold.txt', os.path.join(FINAL_SAVE_PATH, 'gold_spider_realistic.txt'))
print(f"💾 Đã sao lưu các tệp kết quả dự đoán về Drive: {FINAL_SAVE_PATH}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.7 MB/s eta 0:00:00


✅ Đã whitelist Numpy cho PyTorch.
✅ Đã ép buộc torch.load(weights_only=False) thành công.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>>> [1/5] Kiểm tra và tải dữ liệu Spider...
Đang tải tập Spider-Realistic...
>>> [2/5] Đang xử lý cấu trúc Schema đồng bộ...

>>> [3/5] Đang tải mô hình CodeT5 + LoRA lên cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Tìm thấy: 508 mẫu thử nghiệm trong tập Spider-Realistic.

>>> [4/5] Bắt đầu sinh câu lệnh SQL (Batch Processing)...
Tiến độ: 508/508
✅ Quá trình dịch thuật hoàn tất trong 74.06 giây!

>>> [5/5] KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP SPIDER-REALISTIC:
💾 Đã sao lưu các tệp kết quả dự đoán về Drive: /content/drive/MyDrive/CodeT5-small_LoRA


In [ ]:
import os
import subprocess

print("\n>>> [5/5] Kiểm tra và Tiến hành Đánh giá Chi tiết:")

# Bước 1: Kiểm tra xem file pred.txt có dữ liệu hay không để loại trừ lỗi Inference
if os.path.exists('pred.txt'):
    with open('pred.txt', 'r', encoding='utf-8') as f:
        pred_lines = f.readlines()
    print(f"📝 File pred.txt chứa: {len(pred_lines)} câu lệnh SQL được sinh ra.")
else:
    print("❌ Lỗi: Không tìm thấy file pred.txt từ bước Inference!")

# Bước 2: Khắc phục lỗi mã hóa ký tự đặc biệt của file evaluation.py gốc
with open("evaluation.py", "r", encoding="utf-8") as f:
    eval_content = f.read()

if 'text_factory' not in eval_content:
    eval_content = eval_content.replace(
        'conn = sqlite3.connect(db)',
        'conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")'
    )
    with open("evaluation.py", "w", encoding="utf-8") as f:
        f.write(eval_content)
    print("✅ Đã cấu hình text_factory thành công cho bộ chấm điểm.")

# Bước 3: Sử dụng subprocess thay thế os.system để giải quyết triệt để lỗi nuốt luồng văn bản
cmd = [
    "python", "evaluation.py",
    "--gold", "gold.txt",
    "--pred", "pred.txt",
    "--db", "spider_data/database",
    "--table", "spider_data/tables.json",
    "--etype", "all"
]

print("🚀 Đang chạy script chấm điểm chính thức từ phòng thí nghiệm Spider...")
result = subprocess.run(cmd, capture_output=True, text=True)

# Bước 4: Ép hiển thị toàn bộ kết quả xuất ra màn hình Colab
if result.stdout:
    print("\n📊 BẢNG KẾT QUẢ ĐÁNH GIÁ ĐỘ CHÍNH XÁC (OFFICIAL RESULTS):")
    print("-" * 60)
    print(result.stdout)
    print("-" * 60)

# Hiển thị lỗi hệ thống nếu tiến trình chấm điểm gặp trục trặc
if result.stderr:
    print("\n⚠️ Nhật ký cảnh báo/lỗi hệ thống (nếu có):")
    print(result.stderr)


>>> [5/5] Kiểm tra và Tiến hành Đánh giá Chi tiết:
📝 File pred.txt chứa: 508 câu lệnh SQL được sinh ra.
🚀 Đang chạy script chấm điểm chính thức từ phòng thí nghiệm Spider...

📊 BẢNG KẾT QUẢ ĐÁNH GIÁ ĐỘ CHÍNH XÁC (OFFICIAL RESULTS):
------------------------------------------------------------
medium pred: SELECT name ,  country ,  age FROM singer ORDER BY age DESC LIMIT 1
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC

medium pred: SELECT song_name ,  song_release_year FROM singer ORDER BY age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT song_name ,  song_release_year FROM singer ORDER BY age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT count(*) ,  T1.name FROM singer AS T1 JOIN singer_in_concert AS T2 ON T1.singer_id  =  T2.singer_id GROUP BY T1.name
medium gold: SELECT country ,  count(*) FROM singer GROUP BY count